In [ ]:
# =============================================================================
# jobs/29_load_census_deprivation.py
# Census 2021 by output area and the English Indices of Deprivation 2025 by LSOA, for the exposure grid
# (plan: docs/EXPOSURE_AND_RISK_GRID_PLAN.md). Public data, so the tables go in the unrestricted schema.
#
# OUTPUT  config.TBL_CENSUS_OA   oa, population, households, age65plus, share_65plus          (England output areas)
#         config.TBL_IMD_LSOA    lsoa, imd_score, imd_rank, imd_decile, domain deciles         (England LSOAs)
# Both are overwritten in full. Manual: re-run when the source files change (the Census is one-off; the IoD about every three years).
#
# WHERE THE DATA COMES FROM
#   Census 2021: the Nomis bulk tables census2021-ts001 (usual residents), ts007a (age, five-year bands) and ts041 (households). Each
#   zip holds one CSV per geography; the output-area one is used. Downloaded unless local_dir holds the zips (named as Nomis names them).
#   IoD2025: the gov.uk statistics page is read for the link to the "all ranks, scores, deciles" file (File 7). If that fails, paste the
#   file's address, or a path in a volume, into iod_url.
#
# WHAT TO CHECK IN THE OUTPUT (the notebook prints these)
#   the columns it used for each figure; how far the age table's total differs from the population table (small differences are expected, because
#   Census 2021 perturbs each table separately: it should be a few people at most, with a net near zero);
#   about 178,000 England output areas and 33,755 England LSOAs; the IMD deciles run 1 to 10.
#
# Widgets: local_dir, nomis_base, iod_page, iod_url, england_only.
# =============================================================================

import os
import sys

_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

import importlib
import io
import pandas as pd
import requests

import config
import importlib as _il

_il.reload(config)   # a Python session that ran an earlier version of config.py would otherwise miss tables added since
import utils.census_load as _census_load
importlib.reload(_census_load)
from utils.census_load import census_oa_table, find_iod_links, iod_lsoa_table, read_nomis_zip
from utils.helpers import get_spark, utc_now

spark = get_spark()

dbutils.widgets.text("local_dir", "", "Folder holding the Nomis zips (optional)")
dbutils.widgets.text("nomis_base", "https://www.nomisweb.co.uk/output/census/2021/", "Nomis bulk download folder")
dbutils.widgets.text("iod_page", "https://www.gov.uk/government/statistics/english-indices-of-deprivation-2025", "gov.uk page for the IoD2025")
dbutils.widgets.text("iod_url", "", "IoD2025 File 7 address or volume path (optional)")
dbutils.widgets.dropdown("england_only", "true", ["true", "false"], "Keep England only")
local_dir = dbutils.widgets.get("local_dir").strip()
nomis_base = dbutils.widgets.get("nomis_base").strip().rstrip("/") + "/"
iod_page = dbutils.widgets.get("iod_page").strip()
iod_url = dbutils.widgets.get("iod_url").strip()
england_only = dbutils.widgets.get("england_only") == "true"

HEADERS = {"User-Agent": "FGS-census-load"}


def fetch_bytes(url_or_path):
    if url_or_path.startswith("http"):
        r = requests.get(url_or_path, timeout=300, headers=HEADERS)
        r.raise_for_status()
        return r.content
    with open(url_or_path, "rb") as f:
        return f.read()


def write_table(pdf, table, comment):
    out = pdf.copy()
    out["loaded_at"] = utc_now().isoformat()
    spark.createDataFrame(out).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    spark.sql(f"COMMENT ON TABLE {table} IS '{comment}'")
    print(f"Written {len(out):,} rows to {table}")

In [ ]:
# =============================================================================
# CENSUS 2021, BY OUTPUT AREA
# =============================================================================

frames = {}
for table in ("ts001", "ts007a", "ts041"):
    name = f"census2021-{table}.zip"
    src = os.path.join(local_dir, name) if local_dir else nomis_base + name
    print(f"Reading {src}")
    frames[table] = read_nomis_zip(fetch_bytes(src), "oa")
    print(f"   {table}: {len(frames[table]):,} rows, columns {list(frames[table].columns)[:4]} ...")

census, crep = census_oa_table(frames["ts001"], frames["ts007a"], frames["ts041"])
print("\nColumns used:", {k: v for k, v in crep.items() if k.endswith("column") or k == "age_columns"})
print(f"Output areas: {crep['rows']:,}, of which England {crep['england_rows']:,}.")
print(f"Age table total against the population table: differs in {crep['age_total_differs_from_population_in']:,} areas, by more than 5 in {crep['age_total_differs_by_more_than_5']:,}, "
      f"largest difference {crep['age_total_max_difference']:,}, net {crep['age_total_difference_in_people']:,} people. "
      "Census 2021 perturbs each table separately, so small differences are expected; a large maximum or a big net figure would mean a column is wrong.")
if england_only:
    census = census[census["oa"].astype(str).str.startswith("E")].reset_index(drop=True)
print(f"Population (kept): {int(census['population'].sum()):,}   households: {int(census['households'].sum()):,}   aged 65 or over: {int(census['age65plus'].sum()):,} "
      f"({census['age65plus'].sum() / census['population'].sum():.1%})")
display(census.head(10).astype(str))
write_table(census, config.TBL_CENSUS_OA,
            "Census 2021 by output area (ONS via Nomis, Open Government Licence): usual residents (ts001), households (ts041) and people aged 65 and over (ts007a, bands from 65 to 69). England only unless the notebook was run otherwise. Job 29.")

In [ ]:
# =============================================================================
# ENGLISH INDICES OF DEPRIVATION 2025, BY LSOA
# =============================================================================

candidates = []
if not iod_url:
    page = requests.get(iod_page, timeout=60, headers=HEADERS)
    page.raise_for_status()
    candidates = find_iod_links(page.text)
    print("Links on the page that look like File 7, best first:")
    for u in candidates[:6]:
        print("  ", u)
    if not candidates:
        raise Exception("No csv, xlsx or ods link found on the gov.uk page. Paste the File 7 address into the iod_url widget.")
    iod_url = candidates[0]
print(f"Reading {iod_url}")
content = fetch_bytes(iod_url)
raw = pd.read_csv(io.BytesIO(content)) if iod_url.lower().endswith(".csv") else pd.read_excel(io.BytesIO(content))
print(f"   {len(raw):,} rows, {len(raw.columns)} columns")

imd, irep = iod_lsoa_table(raw)
print("\nColumns used:", {k: irep[k] for k in ("code_column", "decile_column")})
print(f"LSOAs: {irep['rows']:,}, England {irep['england_lsoas']:,} (expected 33,755). Decile values: {irep['decile_values']}")
if england_only:
    imd = imd[imd["lsoa"].astype(str).str.startswith("E01")].reset_index(drop=True)
print("Deciles (1 is the most deprived 10 per cent of LSOAs):")
display(imd["imd_decile"].value_counts().sort_index().rename("lsoas").reset_index().astype(str))
write_table(imd, config.TBL_IMD_LSOA,
            "English Indices of Deprivation 2025 by LSOA (MHCLG, Open Government Licence): IMD score, rank (1 most deprived) and decile (1 most deprived 10 per cent), and domain deciles where present. Not comparable with the 2019 indices. Job 29.")


In [ ]:
# Exit is in its own cell: in an interactive run Databricks replaces a cell's output with the exit message,
# so the checks above would be hidden if this line shared their cell.
dbutils.notebook.exit("success")
